## Explore Dataset

In [1]:
import geopandas as gpd
import shapely
import numpy as np
import sys
from get_and_transform_data import get_and_transform_data
import RiskMap

# Configure logger
import logging

logger = logging.getLogger("Exploration")
logger.setLevel(logging.DEBUG)
handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

## Obtaining data

In [2]:
cso_all, wwtw_all, d, rivers_all = get_and_transform_data()

2025-05-30 08:43:36,236 - INFO - Loading downloaded and transformed data...
2025-05-30 08:43:54,599 - INFO - Data loaded


In [3]:
# Add the catchment geojson url of your choosing here to clip the data to that catchment
cmnt = gpd.read_file(
    "https://environment.data.gov.uk/catchment-planning/OperationalCatchment/3052.geojson"
)  # Browney
cmnt.to_crs(cso_all.crs, inplace=True)

### Clip to Catchment Data

In [4]:
rivers = gpd.clip(rivers_all, cmnt)
# Remove all rivers that aren't LineStrings after Clipping
rivers = rivers[
    rivers.geometry.apply(lambda x: isinstance(x, shapely.LineString))
]
cso = gpd.clip(cso_all, cmnt)
wwtw = gpd.clip(wwtw_all, cmnt)

## Construct Risk Info Map

In [5]:
RIVER_RESOLUTION: float = 200
MAX_OUTFALL_DIST_FROM_RIVER: float = 700
MAX_RISK_DISTANCE: float = np.inf

In [6]:
# Generate Graph from Rivers
riskmap = RiskMap.RiskMap(rivers, "RivID", RIVER_RESOLUTION, directed=True)
riskmap.construct_river_graph()

In [7]:
# Add CSO information
outfallLocations = riskmap.add_point_info_to_map(
    cso.index,
    cso["geometry"],
    "csoInfo",
    MAX_RISK_DISTANCE,
    MAX_OUTFALL_DIST_FROM_RIVER,
)
cso = cso.assign(outfallLocation=outfallLocations)

In [8]:
# Add WwTW information
outfallLocations = riskmap.add_point_info_to_map(
    wwtw.index,
    wwtw["geometry"],
    "wwtwInfo",
    MAX_RISK_DISTANCE,
    MAX_OUTFALL_DIST_FROM_RIVER,
)
wwtw = wwtw.assign(outfallLocation=outfallLocations)

In [9]:
# Import the risk method
from utils import consentsRisk

In [10]:
# Construct and get the GDF
riskmap.add_all_risk(
    [
        {
            "info_col": "csoInfo",
            "risk_name": "csoRisk",
            "risk_method": consentsRisk.add_cso_p_risk,
            "aggregation_method": "w_avg",
            "distance_scaling": consentsRisk.p_distance_scaling,
        },
        {
            "info_col": "wwtwInfo",
            "risk_name": "wwtwRisk",
            "risk_method": consentsRisk.add_wwtw_p_risk,
            "aggregation_method": "w_avg",
            "distance_scaling": consentsRisk.p_distance_scaling,
        },
    ]
)

In [11]:
riskmap.construct_risk_map()
river_points = riskmap.get_risk_map()

## Explore

In [12]:
wwtw = wwtw.join(consentsRisk.get_wwtw_values(), how="left", validate="1:1")
cso = cso.join(consentsRisk.get_cso_values(), how="left", validate="1:1")
cso["SpillDurationStr"] = cso["SpillDuration"].astype(str)

In [13]:
csocols = [
    "geometry",
    "outfallLocation",
    "Site Name\n(EA Consents Database)",
    "SpillDurationStr",
    "Counted spills using 12-24h count method",
    "Weir Setting",
]

wwtwcols = [
    "geometry",
    "outfallLocation",
    "DISCHARGE_SITE_NAME",
    "P",
    "DWF",
]

csolines = cso.apply(
    lambda x: shapely.LineString(
        [x["geometry"], x["outfallLocation"]]
        if x["geometry"] is not None and x["outfallLocation"] is not None
        else None
    ),
    axis=1,
).set_crs(cso.crs)
wwtwlines = wwtw.apply(
    lambda x: shapely.LineString(
        [x["geometry"], x["outfallLocation"]]
        if x["geometry"] is not None and x["outfallLocation"] is not None
        else None
    ),
    axis=1,
).set_crs(cso.crs)

In [14]:
# Plot the catchment
m = cmnt.explore(
    highlight=False,
    tooltip=False,
    popup=False,
    color="grey",
)

# Plot the rivers
rivers.explore(
    m=m,
    color="blue",
)
riskmap.get_rivers_split().explore(
    m=m, color="darkblue", style_kwds={"dashArray": "5, 5"}
)

# Plot the risk
river_points[["geometry", "Risk", "csoRisk", "wwtwRisk"]].explore(
    m=m,
    column="Risk",
    cmap="RdYlGn_r",
    vmax=200,
    marker_kwds={"radius": 5},
)

# Plot the CSOs, WwTWs and connecting lines
cso[csocols].explore(m=m, color="pink", marker_kwds={"radius": 8})
wwtw[wwtwcols].explore(m=m, color="yellow", marker_kwds={"radius": 8})

cso[csocols].set_geometry("outfallLocation").explore(
    m=m, color="red", marker_kwds={"radius": 8}
)
wwtw[wwtwcols].set_geometry("outfallLocation").explore(
    m=m, color="brown", marker_kwds={"radius": 8}
)

csolines.explore(m=m, style_kwds={"color": "red", "dashArray": "5, 5"})
wwtwlines.explore(m=m, style_kwds={"color": "brown", "dashArray": "5, 5"})